# Практика 9. PyTorch с нуля

Курс «Машинное обучение» · Часть 2 · 90 минут

Обучим классификатор одежды по изображениям 28×28. Сначала разберём тензоры
и градиенты, затем соберём MLP вручную и через `nn.Module`.

1. Тензоры и размерности.
2. Autograd и шаг градиентного спуска.
3. FashionMNIST и батчи.
4. MLP вручную и через `nn.Module`.
5. Цикл обучения и сравнение с линейной моделью.
6. Ошибки модели и выводы.

В Colab выберите **Среда выполнения → Выполнить все**. CPU достаточно;
GPU можно включить в настройках среды. Первый запуск скачивает данные.
Время обучения зависит от среды. Для самостоятельной работы заполните задания
перед ячейками самопроверки. Фиксированный seed не гарантирует одинаковые числа
на разных устройствах и версиях PyTorch.

Используем `torch`, `torchvision`, `numpy`, `matplotlib`, `pandas` и `scikit-learn`;
они входят в обычную среду Colab. Для локального запуска:
`python3 -m pip install torch torchvision numpy matplotlib pandas scikit-learn`.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| устройство:', device)

## 1. Тензоры: размерности важнее синтаксиса

`Tensor` похож на массив NumPy, но умеет работать на GPU и хранить историю
операций для вычисления градиентов. Для батча изображений используем форму
`[batch, channels, height, width]`, для входа полносвязного слоя — `[batch, features]`.

### Задание 1. Предсказание для батча

Создайте `x` из чисел от 0 до 23 с формой `(2, 1, 3, 4)` и типом `float32`.
Разверните только измерения после батча: `flat = x.flatten(start_dim=1)`.
Создайте `w` из единиц формы `(12, 3)` и `b = [0., 1., 2.]`.
Посчитайте `logits = flat @ w + b`. Сложение добавит один вектор ко всем строкам:
это broadcasting.

In [ ]:
# Создайте x, flat, w, b и logits.
x = None
flat = None
w = None
b = None
logits = None

In [ ]:
assert x.shape == (2, 1, 3, 4) and x.dtype == torch.float32
assert flat.shape == (2, 12)
expected = torch.stack([flat.sum(1) + offset for offset in range(3)], dim=1)
assert torch.equal(logits, expected) and logits.shape == (2, 3)
print('Размерности и broadcasting: OK')

## 2. Autograd: от производной к обновлению веса

Для `f(a) = (a - 3)²` производная равна `2(a - 3)`. При `a = 1` она отрицательная:
шаг `a ← a - lr × grad` увеличит `a` и приблизит его к минимуму.
`backward()` записывает производную в `.grad` и при повторном вызове добавляет
новый градиент к старому.

### Задание 2. Один шаг без оптимизатора

Создайте скаляр `a = 1.` с `requires_grad=True`, вычислите `loss_before`,
вызовите `backward()` и сохраните копию градиента в `grad_before`.
Внутри `with torch.no_grad():` обновите `a` с `lr=0.1`.
Обнулите `a.grad`, затем посчитайте `loss_after`.
Изменение параметра не должно само становиться частью графа вычислений.

In [ ]:
# Посчитайте производную и сделайте один шаг.
a = None
grad_before = None
loss_before = None
loss_after = None

In [ ]:
assert torch.allclose(grad_before, torch.tensor(-4.))
assert torch.allclose(a.detach(), torch.tensor(1.4))
assert loss_after.item() < loss_before.item()
assert a.grad.item() == 0.
print('Градиент и направление шага: OK')

In [ ]:
z = torch.tensor(2., requires_grad=True)
(z ** 2).backward()
first_grad = z.grad.clone()
(z ** 2).backward()  # Новый граф, но та же переменная z и накопленный .grad.
print('После первого backward:', first_grad.item())
print('После второго backward:', z.grad.item())
assert torch.equal(z.grad, 2 * first_grad)
z.grad.zero_()

### Вопрос

Зачем нужен `optimizer.zero_grad()` перед следующим батчем?

_Ваш ответ:_



## 3. FashionMNIST: разделяем данные до обучения

В наборе 60 000 обучающих и 10 000 тестовых изображений, десять классов одежды.
Для занятия используем 12 000 обучающих и 3 000 валидационных примеров,
выбранных со стратификацией из исходной обучающей части. Остальные 45 000
не используем, чтобы сократить время занятия. Тестовую часть откроем только
после сравнения моделей по валидации.

`ToTensor()` переводит пиксели из `[0, 255]` в `float32` из `[0, 1]`.
Параметров, обучаемых на данных, у этого преобразования нет.
Данные скачиваются в `~/Downloads/ml-course-data`.
Нужен доступ к официальному серверу FashionMNIST.

In [ ]:
data_root = Path.home() / 'Downloads' / 'ml-course-data'
train_full = FashionMNIST(data_root, train=True, download=True, transform=ToTensor())
indices = np.arange(len(train_full))
train_idx, rest_idx = train_test_split(
    indices, train_size=12000, stratify=train_full.targets.numpy(), random_state=SEED)
val_idx, _ = train_test_split(
    rest_idx, train_size=3000, stratify=train_full.targets.numpy()[rest_idx],
    random_state=SEED)
train_ds = Subset(train_full, train_idx)
val_ds = Subset(train_full, val_idx)
classes = ['Футболка', 'Брюки', 'Свитер', 'Платье', 'Пальто',
           'Сандалии', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботинки']
assert set(train_idx).isdisjoint(val_idx)
print('Обучение:', len(train_ds), '| валидация:', len(val_ds))
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, (image, target) in zip(axes.flat, [train_ds[i] for i in range(10)]):
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(classes[target])
    ax.axis('off')
plt.tight_layout()
plt.show()

### Задание 3. Соберите загрузчики батчей

Создайте `train_loader` и `val_loader` с `batch_size=128` и `num_workers=0`.
Для обучения включите `shuffle=True` и передайте
`generator=torch.Generator().manual_seed(SEED)`. Для валидации перемешивание
не требуется. Извлеките один батч в `xb, yb`.
Метки классов должны быть целыми числами, а не one-hot векторами.

In [ ]:
train_loader = None
val_loader = None
# xb, yb = ...

In [ ]:
assert xb.shape == (128, 1, 28, 28) and xb.dtype == torch.float32
assert yb.shape == (128,) and yb.dtype == torch.int64
assert 0 <= xb.min() <= xb.max() <= 1
assert 0 <= yb.min() <= yb.max() < 10
assert len(train_loader.dataset) == 12000 and len(val_loader.dataset) == 3000
print('Батчи: OK')

## 4. Один MLP двумя способами

Сеть: `784 → 64 → ReLU → 10`. Сначала запишем прямой проход через умножения
матриц и обучаемые тензоры. Производные по этим тензорам посчитает autograd;
ручной backprop остаётся для домашнего задания.

### Задание 4. Прямой проход вручную

Создайте `W1`, `b1`, `W2`, `b2` как `nn.Parameter`. Формы весов: `(784, 64)`
и `(64, 10)`, смещений: `(64,)` и `(10,)`. Веса инициализируйте
`torch.randn(...) * 0.01`, смещения — нулями. Напишите `manual_forward(images)`:
развернуть батч, применить первый линейный слой, `torch.relu`, второй слой.
На выходе нужны **логиты**, то есть произвольные числа перед softmax.
Вычислите `manual_logits = manual_forward(xb)`, затем потери
`nn.CrossEntropyLoss()(manual_logits, yb)` и вызовите `backward()`.

In [ ]:
# Создайте W1, b1, W2, b2.
def manual_forward(images):
    raise NotImplementedError('Реализуйте два линейных слоя и ReLU')

In [ ]:
assert manual_forward(xb).shape == (len(xb), 10)
for parameter in (W1, b1, W2, b2):
    assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
assert sum(p.numel() for p in (W1, b1, W2, b2)) == 50890
print('Прямой и обратный проход: OK')

### Задание 5. Та же сеть через nn.Module

Определите класс `MLP(nn.Module)`. В `__init__` вызовите `super().__init__()`
и создайте `self.net = nn.Sequential(...)` из `Flatten`, `Linear(784, 64)`,
`ReLU`, `Linear(64, 10)`. В `forward` верните `self.net(images)`.
Создайте `equivalent = MLP()` для проверки совпадения с ручным проходом.

`nn.Linear` хранит веса в форме `[out_features, in_features]`, поэтому
при копировании из ручной реализации веса нужно транспонировать.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Создайте self.net.

    def forward(self, images):
        raise NotImplementedError('Верните результат self.net')

equivalent = MLP()

In [ ]:
with torch.no_grad():
    equivalent.net[1].weight.copy_(W1.T)
    equivalent.net[1].bias.copy_(b1)
    equivalent.net[3].weight.copy_(W2.T)
    equivalent.net[3].bias.copy_(b2)
assert torch.allclose(equivalent(xb), manual_forward(xb), atol=1e-6)
assert sum(p.numel() for p in equivalent.parameters()) == 50890
print('Две реализации дают одинаковый результат: OK')

### Вопрос

Почему перед `CrossEntropyLoss` не нужен `Softmax`? Что изменится, если убрать ReLU между двумя линейными слоями?

_Ваш ответ:_



## 5. Цикл обучения

На каждом батче: обнулить градиенты → получить логиты → посчитать потери →
`backward()` → `step()`. Параметры и данные должны находиться на одном устройстве.

### Задание 6. Обучение одной эпохи

Заполните `train_epoch(model, loader, optimizer, criterion)`.
Включите `model.train()`, перенесите батч на `device` и выполните пять действий.
Возвращайте среднюю потерю на объект, а не среднее невзвешенное по батчам:
последний батч может быть меньше остальных. Образец такой агрегации
есть в готовой функции `evaluate` ниже.

In [ ]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0., 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss_sum += criterion(outputs, targets).item() * len(targets)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += len(targets)
    return loss_sum / total, correct / total

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    raise NotImplementedError('Реализуйте обучение одной эпохи')

In [ ]:
# Два батча ловят забытый zero_grad; неполный батч проверяет усреднение.
torch.manual_seed(SEED)
probe = MLP().to(device)
reference = MLP().to(device)
reference.load_state_dict(probe.state_dict())
probe_optimizer = torch.optim.SGD(probe.parameters(), lr=0.1)
reference_optimizer = torch.optim.SGD(reference.parameters(), lr=0.1)
probe_loader = DataLoader(Subset(train_ds, range(150)), batch_size=128)
probe_loss = train_epoch(probe, probe_loader, probe_optimizer, criterion)
reference.train()
reference_sum = 0.
for images, targets in probe_loader:
    images, targets = images.to(device), targets.to(device)
    reference_optimizer.zero_grad()
    reference_loss = criterion(reference(images), targets)
    reference_loss.backward()
    reference_optimizer.step()
    reference_sum += reference_loss.item() * len(targets)
assert np.isclose(probe_loss, reference_sum / 150, rtol=1e-5)
for actual, expected in zip(probe.parameters(), reference.parameters()):
    assert torch.allclose(actual, expected, atol=1e-6), 'Проверьте zero_grad и шаг обновления'
    assert actual.grad is not None and torch.isfinite(actual.grad).all()
print('Два шага, обнуление градиентов и средняя потеря: OK')

### Сравнение с линейной моделью

Обучим `Flatten → Linear(784, 10)` и MLP по пять эпох с Adam и `lr=0.001`.
Каждая модель получает одинаковый порядок батчей. Выбор победителя делаем
по валидационной accuracy; test пока не используем.
Это сравнение при фиксированных настройках, а не доказательство, что одна
архитектура всегда лучше другой.

In [ ]:
EPOCHS = 5
models, histories, rows = {}, {}, []
for name in ['Линейная', 'MLP']:
    torch.manual_seed(SEED)
    model = (nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
             if name == 'Линейная' else MLP()).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0,
                        generator=torch.Generator().manual_seed(SEED))
    history = []
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_loss': val_loss, 'val_acc': val_acc})
        print(f'{name}, эпоха {epoch}: train={train_loss:.3f}, '
              f'val={val_loss:.3f}, accuracy={val_acc:.3f}')
    models[name], histories[name] = model, pd.DataFrame(history)
    rows.append({'Модель': name, 'Параметров': sum(p.numel() for p in model.parameters()),
                 'val_loss': val_loss, 'val_accuracy': val_acc})
results = pd.DataFrame(rows).set_index('Модель')
display(results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history.epoch, history.train_loss, label=f'{name}: train')
    axes[0].plot(history.epoch, history.val_loss, '--', label=f'{name}: val')
    axes[1].plot(history.epoch, history.val_acc, marker='o', label=name)
axes[0].set(xlabel='Эпоха', ylabel='Cross-entropy', title='Потери')
axes[1].set(xlabel='Эпоха', ylabel='Accuracy', title='Качество на валидации')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
for history in histories.values():
    assert np.isfinite(history[['train_loss', 'val_loss', 'val_acc']]).all().all()
    assert history.train_loss.iloc[-1] < history.train_loss.iloc[0]
assert results.val_accuracy.between(0, 1).all()
best_name = results.val_accuracy.idxmax()
print('Выбрана по валидации:', best_name)

### Вопрос

У какой модели выше validation accuracy? Что видно по кривым? Можно ли по этим пяти эпохам уверенно утверждать, что модель переобучилась?

_Ваш ответ:_



## 6. Финальная проверка и ошибки

Теперь один раз оценим выбранную модель на всех 10 000 тестовых изображениях.
После просмотра теста не подбирайте по нему настройки. Для дальнейших
экспериментов вернитесь к train/validation.
Матрица ошибок покажет, какие классы путаются, а изображения помогут понять причину.

In [ ]:
test_ds = FashionMNIST(data_root, train=False, download=True, transform=ToTensor())
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
best_model = models[best_name]
test_loss, test_acc = evaluate(best_model, test_loader)
print(f'{best_name}: test loss={test_loss:.4f}, test accuracy={test_acc:.4f}')
predictions, targets = [], []
best_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        predictions.append(best_model(images.to(device)).argmax(1).cpu())
        targets.append(labels)
predictions = torch.cat(predictions).numpy()
targets = torch.cat(targets).numpy()
assert len(predictions) == 10000
assert np.isclose((predictions == targets).mean(), test_acc)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(
    targets, predictions, labels=range(10), display_labels=classes,
    xticks_rotation=45, cmap='Blues', colorbar=False, ax=ax)
ax.set_title(f'{best_name}: ошибки на тесте')
plt.tight_layout()
plt.show()

In [ ]:
errors = np.flatnonzero(predictions != targets)[:10]
if len(errors):
    fig, axes = plt.subplots(2, 5, figsize=(13, 6))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, errors):
        image, target = test_ds[int(index)]
        ax.imshow(image.squeeze(), cmap='gray')
        ax.set_title(f'Верно: {classes[target]}\nПрогноз: {classes[predictions[index]]}', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Ошибок на тесте нет.')
print(f'Итог: выбрана {best_name}; test accuracy = {test_acc:.3f}.')

### Вопрос

Назовите два класса, которые модель часто путает, и приведите одно объяснение по изображениям. Что проверите в следующем эксперименте?

_Ваш ответ:_



## Что должно получиться

Вы умеете проверять формы тензоров, получать градиенты, собирать MLP,
обучать его батчами и оценивать на отложенных данных. Сохраните таблицу
сравнения, кривые и свой разбор ошибок.

Источники: [PyTorch Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html),
[autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html),
[FashionMNIST](https://github.com/zalandoresearch/fashion-mnist).
Здесь обучение строится по той же схеме, что в лекции 9; CNN и приёмы
стабилизации обучения будут в следующих занятиях.